# Transfer Learning using VGG19

## What is Transfer Learning?

Transfer learning means using a model that was already trained on a large dataset and adapting it to a new task. For CIFAR10, we use VGG19 pretrained on ImageNet, then add new layers to classify our 10 target classes.

Benefits of transfer learning:

* saves training time
* works well with smaller datasets
* reuses useful visual features like edges and textures

## VGGNet

VGGNet is a famous deep convolutional neural network developed by researchers at Visual Geometry Group at the University of Oxford. It was introduced in 2014 and became one of the most influential CNN architectures in deep learning history.

**Historical Background**

Before VGGNet:

* CNN architectures were becoming deeper,
* but designs were complicated.

VGGNet showed that:

> Very deep networks with small filters can achieve excellent performance.

Its design philosophy was:

* simplicity
* depth
* uniform architecture

**Versions of VGGNet**

Popular versions:

| Model | Layers    |
| ----- | --------- |
| VGG11 | 11 layers |
| VGG13 | 13 layers |
| VGG16 | 16 layers |
| VGG19 | 19 layers |

The most famous is: VGG16

### Why VGGNet Was Revolutionary

VGGNet introduced several key ideas:

| Innovation               | Importance                |
| ------------------------ | ------------------------- |
| Very deep CNN            | Improved feature learning |
| Small 3×3 filters        | Reduced parameters        |
| Uniform architecture     | Simpler design            |
| Deep feature hierarchies | Better abstraction        |

### Main Idea of VGGNet

Instead of large filters like 11×11 in AlexNet

VGGNet uses:
$$
3\times3
$$
filters everywhere.

### Why Small 3×3 Filters?

Using multiple 3×3 convolutions gives benefits:

Two 3×3 convolutions approximate a:

$$
5\times5
$$

receptive field.

Three 3×3 convolutions approximate a:
$$
7\times7
$$

receptive field.


![CNN Architecture](https://raw.githubusercontent.com/masum184e/handbook-old/removed/images/cnn/vggnet.png)

image source: https://raw.githubusercontent.com/masum184e/handbook-old/removed/images/cnn/vggnet.png

### Input Layer (Original VGG19 on ImageNet)

Standard VGGNet uses ImageNet-sized inputs:

$$
224 \times 224 \times 3
$$

* RGB images
* Fixed input size (important for FC layers later)

---

## **CIFAR10 Adaptation**

**Important:** CIFAR10 has different image dimensions than ImageNet.

| Property      | ImageNet | CIFAR10 | Our Setup |
| ------------- | -------- | ------- | --------- |
| Original Size | 224×224  | 32×32   | 32×32     |
| After Upsample| -        | -       | 64×64     |
| Channels      | 3 (RGB)  | 3 (RGB) | 3 (RGB)   |

**Why Upsample?**
* VGG19 was trained on 224×224 ImageNet images
* CIFAR10 images are 32×32 (much smaller)
* Upsampling to 64×64 helps preserve more information
* Trade-off: More computation vs. better feature extraction

---

### Block 1 (Conv Block 1) - After Upsampling to 64×64

| Layer | Type        | Details         | Output    |
| ----- | ----------- | --------------- | --------- |
| C1_1  | Conv + ReLU | 3×3, 64 filters | 64×64×64  |
| C1_2  | Conv + ReLU | 3×3, 64 filters | 64×64×64  |
| S1    | Max Pool    | 2×2, stride 2   | 32×32×64  |

**What it learns:**

* edges
* simple textures
* color gradients


## Importing libraries

In [ ]:
import tensorflow as tf
import keras
from keras import layers
from keras.utils import to_categorical
from keras.applications.vgg19 import VGG19

import numpy as np
import matplotlib.pyplot as plt

: 

## Load the Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

: 

In [4]:
print(f'Shape of x_train: {x_train.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of x_test: {x_test.shape}')
print(f'Shape of y_test: {y_test.shape}')

## Normalize Image Data

In [5]:
x_train = x_train.astype("float32") / 255.0 # 
x_test = x_test.astype("float32") / 255.0

## One-Hot Encoding

In [6]:
num_classes = len(np.unique(y_train))  

# Convert labels to one-hot encoding
# Example: class 3 -> [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train, num_classes)  # Convert training labels to one-hot vectors
y_test = to_categorical(y_test, num_classes)    # Convert test labels to one-hot vectors

## Preparing the base model

In [7]:
# Load pretrained VGG19 model
base_model = VGG19(
  weights='imagenet',              # Load weights trained on ImageNet (ImageNet has 1000 classes)
  include_top=False,               # Remove top classification layers (we will add custom layers for CIFAR10)
  input_shape=(64, 64, 3)          # Input shape: 64×64×3 (upsampled from CIFAR10's 32×32)
)

# Freeze base model weights - don't update them during training
# This is the key idea of transfer learning: use features learned from ImageNet
base_model.trainable = False       # Keep pretrained weights fixed, only train new custom layers

In [8]:
base_model.summary()

## Transfer Learning using VGG19

In [9]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.UpSampling2D(size=(2, 2)),  # this increases the dimention to 64 x 64
    base_model,

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),

    layers.Dense(10, activation='softmax'),
])

model.summary()

## Compile the model

In [10]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Train the model

In [ ]:
history = model.fit(
    x_train,
    y_train,
    batch_size=64,           # Process 64 samples at a time before updating weights
    epochs=3,                # Train for 3 complete passes through the dataset
    validation_split=0.1,    # Use 10% of training data (5000 samples) to validate during training
)
# Returns training history with loss and accuracy metrics for each epoch

Epoch 1/3
291/704 ━━━━━━━━━━━━━━━━━━━━ 1:01:35 9s/step - accuracy: 0.4131 - loss: 1.6673

## Visualization

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
import random

idx = random.randint(0, len(x_test) - 1)

prediction_probabilities = model.predict(x_test[idx:idx+1], verbose=0)

predicted_class_idx = np.argmax(prediction_probabilities)
actual_class_idx = np.argmax(y_test[idx])

predicted_label = class_names[predicted_class_idx]
actual_label = class_names[actual_class_idx]

plt.figure(figsize=(3, 3))
plt.imshow(x_test[idx])
plt.title(f"Pred: {predicted_label} | Actual: {actual_label}", fontsize=10)
plt.axis("off")
plt.show()

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # Tools for computing and plotting confusion matrix

y_pred = model.predict(x_test)  # Model predictions for all test images

y_pred_classes = np.argmax(y_pred, axis=1)  # Convert predicted probabilities to class labels
y_true = np.argmax(y_test, axis=1)          # Convert one-hot test labels back to class indices

cm = confusion_matrix(y_true, y_pred_classes)  # Compute confusion matrix
# Create a visual display of the confusion matrix with class labels
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(10, 10))  # Plot size for readability
disp.plot(ax=ax, cmap=plt.cm.Blues)
plt.xticks(rotation=45)  # Rotate x labels so class names are easier to read
plt.show()

## Saving the model

In [ ]:
model.save('outputs/vgg19_model.keras')